# BM25、向量检索与 RRF 对照

这份笔记复用你已经跑通的南瓜书语料和 `search_bm25.py`，不重搭项目。

**使用方法**

1. 把本笔记放到 `rag_real_materials_南瓜书练习/` 目录，和 `search_bm25.py`、`chunks.json` 放在一起。
2. 按顺序逐格运行。若目录里已有 `chunk_vectors.npz`，会直接复用，不必重算 857 条向量。
3. **本轮只运行到「四题三路对照」**，看懂输出即可。标了「下一轮」的格子先不要做。
4. 本轮不必调用聊天模型生成答案，也不必上传 GitHub。

结构是：中文说明 → 可运行代码 → 输出该怎么读 → 下一轮的修改实验。


## 0. 检查路径和语料

先确认读到的是同一份 857 条 `chunks.json`。后面的 BM25 会直接调用已有的 `load_chunks`、`build_index`、`search`，不改这些函数的参数。


In [1]:
from pathlib import Path
import sys

# 笔记就在练习目录里时，不用改。
# 若从别的工作目录打开，把路径改成 search_bm25.py 所在文件夹。
DATA_DIR = Path.cwd()
if not (DATA_DIR / "search_bm25.py").exists():
    for cand in [
        DATA_DIR / "rag_real_materials_南瓜书练习",
        DATA_DIR.parent / "rag_real_materials_南瓜书练习",
    ]:
        if (cand / "search_bm25.py").exists():
            DATA_DIR = cand
            break

if not (DATA_DIR / "search_bm25.py").exists() or not (DATA_DIR / "chunks.json").exists():
    raise FileNotFoundError(
        "找不到 search_bm25.py 或 chunks.json。"
        "请把本笔记放到 rag_real_materials_南瓜书练习/ 目录后再运行。"
        f" 当前目录: {Path.cwd()}"
    )

sys.path.insert(0, str(DATA_DIR))
import search_bm25

chunks = search_bm25.load_chunks()
print("练习目录:", DATA_DIR)
print("语料片段数:", len(chunks))
print("示例片段:", chunks[0].get("chunk_id"), "pdf_page=", chunks[0].get("pdf_page"))
if len(chunks) != 857:
    print("注意：片段数不是 857，请确认是否打开了另一份 chunks.json。本轮仍可继续，但结果会和昨天的对照不同。")


练习目录: /Users/ruanwen/PycharmProjects/LocalAgent/human_code/rag_real_materials_南瓜书练习
语料片段数: 857
示例片段: pumpkin-p001-c01 pdf_page= 1


## 1. 回顾：按字面搜，和按意思搜

这一部分你昨天已经跑通过，这里只是在笔记里快速复现，方便和后面的融合结果放在一起看。

- **BM25** 像在书里搜原词：问题和片段里出现相同的字，分就高。所以「哪三种模型评估方法」能打到第 18 页。
- **向量检索** 像按意思找邻居：先把每段话变成一组数字，再看问题和片段的「方向」近不近。用词换了、意思还在，仍可能排到前面。

两种方法都会返回「最像的前几条」，即使书里根本没有答案。有分数不等于找到了答案。

昨天第 4 题的结论（已验收）：换一种说法后，向量检索把 `pumpkin-p018-c03`（第 18 页）排进前 3，BM25 的前 3 没有。这是项目里可以保留的真实对比，但不能据此断言向量全面优于 BM25。


In [2]:
QUESTIONS = [
    "书中介绍了哪三种模型评估方法？",
    "交叉验证法常用来比较哪些效果？",
    "小明的哥哥有什么兴趣爱好？",
    "课本里用哪几种办法来检验模型表现如何？",  # 第 1 题换一种说法
]

bm25_index = search_bm25.build_index(chunks)


def hits_to_rows(hits):
    """把 search() 的 (idx, score, chunk) 转成统一的字典列表。"""
    rows = []
    for idx, score, chunk in hits:
        rows.append({
            "idx": idx,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "pdf_page": chunk["pdf_page"],
            "text": chunk["text"],
        })
    return rows


def search_bm25_rows(query, top_k=3):
    return hits_to_rows(search_bm25.search(bm25_index, chunks, query, top_k=top_k))


def print_rows(title, rows):
    print(f"--- {title} ---")
    if not rows:
        print("  （没有结果）")
        return
    for rank, row in enumerate(rows, start=1):
        print(
            f"  [{rank}] score={row['score']:.3f}  {row['chunk_id']}  pdf_page={row['pdf_page']}"
        )

print("BM25 索引已建立。下面先准备向量检索，再一起打印四题对照。")


Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/9v/q10wx2ks55389kzkfc2mcj9m0000gn/T/jieba.cache
Loading model cost 0.390 seconds.
Prefix dict has been built successfully.


BM25 索引已建立。下面先准备向量检索，再一起打印四题对照。


### 向量怎么来

流程还是同一份切好的片段，不重新分段，也不需要先上向量数据库：

```text
chunks.json（857 个片段）
    ↓ 每个片段的 text 送进同一个 embedding 模型
chunk_vectors.npz（向量 + chunk_id，算一次就缓存）
    ↓ 问题也用同一个模型编码
按余弦相似度取前若干条
```

优先加载已有的 `chunk_vectors.npz`。没有缓存时，先试你现有的本地接口（`base_url=http://192.168.124.18:18082/v1`）是否支持 embeddings；不行再改用免费的中文小模型 `BAAI/bge-small-zh-v1.5`。不要为了这一轮新开通付费接口。


In [5]:
# 从 human_code/.env 读取 EMBED_BASE_URL / EMBED_API_MODEL / OPENAI_API_KEY。
# .env 已在 .gitignore 里，不会入库。必须在下面定义 EMBED_* 的格子之前运行。
from pathlib import Path

from dotenv import load_dotenv

env_path = DATA_DIR.parent / ".env"
if not env_path.is_file():
    env_path = DATA_DIR / ".env"
loaded = load_dotenv(env_path, override=True)
print("已加载 .env:", env_path, "override" if loaded else "(未加载，沿用系统环境变量)")
print("EMBED_BASE_URL =", os.environ.get("EMBED_BASE_URL", "(未设置，将用默认 18082)"))
print("EMBED_API_MODEL =", os.environ.get("EMBED_API_MODEL", "(未设置)"))
print("OPENAI_API_KEY 已设置:", bool(os.environ.get("OPENAI_API_KEY")))


已加载 .env: /Users/ruanwen/PycharmProjects/LocalAgent/human_code/.env override
EMBED_BASE_URL = http://192.168.124.18:8090/v1
EMBED_API_MODEL = qwen3-embedding-0.6b
OPENAI_API_KEY 已设置: True


In [6]:
import os
import numpy as np

CACHE_PATH = DATA_DIR / "chunk_vectors.npz"
EMBED_BASE_URL = os.environ.get("EMBED_BASE_URL", "http://192.168.124.18:18082/v1")
# 若本地接口有 embedding 模型名，写在这里或设环境变量 EMBED_API_MODEL
EMBED_API_MODEL = os.environ.get("EMBED_API_MODEL", "").strip()
LOCAL_MODEL_NAME = "BAAI/bge-small-zh-v1.5"

chunk_ids = [c["chunk_id"] for c in chunks]
_local_encoder = None


def l2_normalize(mat):
    mat = np.asarray(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return mat / norms


def load_vector_cache(path):
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)
    if "vectors" in keys:
        vec_key = "vectors"
    elif "embeddings" in keys:
        vec_key = "embeddings"
    elif len(data.files) == 1:
        vec_key = data.files[0]
    else:
        print("缓存里没有识别到 vectors/embeddings，将重新计算。键名:", data.files)
        return None
    vectors = l2_normalize(data[vec_key])
    if "chunk_ids" in keys:
        cached_ids = [str(x) for x in data["chunk_ids"].tolist()]
        if cached_ids != chunk_ids:
            print("缓存里的 chunk_id 和当前 chunks.json 不一致，将重新计算。")
            return None
    elif vectors.shape[0] != len(chunks):
        print(f"缓存条数是 {vectors.shape[0]}，当前语料是 {len(chunks)} 条，将重新计算。")
        return None
    model_name = None
    if "model" in keys:
        model_name = str(np.asarray(data["model"]).item() if np.asarray(data["model"]).ndim == 0 else data["model"])
    elif "model_name" in keys:
        model_name = str(np.asarray(data["model_name"]).item() if np.asarray(data["model_name"]).ndim == 0 else data["model_name"])
    print(f"已加载向量缓存: {path.name}  shape={vectors.shape}  model={model_name or '未知'}")
    return {"vectors": vectors, "model_name": model_name or "unknown-cache"}


def save_vector_cache(path, vectors, model_name):
    np.savez(
        path,
        vectors=vectors,
        chunk_ids=np.array(chunk_ids),
        model=np.array(model_name),
    )
    print("已写入缓存:", path)


def embed_with_api(texts, model_name):
    from openai import OpenAI
    client = OpenAI()
    client.base_url = EMBED_BASE_URL
    out = []
    batch_size = 32
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        resp = client.embeddings.create(model=model_name, input=batch)
        items = sorted(resp.data, key=lambda x: x.index)
        out.extend([d.embedding for d in items])
        print(f"  API 编码进度 {min(start + batch_size, len(texts))}/{len(texts)}")
    return l2_normalize(out), f"api:{model_name}"


def get_local_encoder():
    global _local_encoder
    if _local_encoder is None:
        from sentence_transformers import SentenceTransformer
        _local_encoder = SentenceTransformer(LOCAL_MODEL_NAME)
    return _local_encoder


def embed_with_local_model(texts):
    model = get_local_encoder()
    vecs = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    return l2_normalize(vecs), LOCAL_MODEL_NAME


def get_or_build_vectors():
    cached = load_vector_cache(CACHE_PATH)
    if cached is not None:
        return cached["vectors"], cached["model_name"]

    texts = [c["text"] for c in chunks]
    if EMBED_API_MODEL:
        print(f"未找到可用缓存，尝试本地接口 embeddings，模型: {EMBED_API_MODEL}")
        try:
            vectors, name = embed_with_api(texts, EMBED_API_MODEL)
            save_vector_cache(CACHE_PATH, vectors, name)
            return vectors, name
        except Exception as exc:
            print("本地 embedding 接口不可用，改用免费中文小模型。原因:", type(exc).__name__, exc)

    print(f"未找到可用缓存，使用本地免费模型 {LOCAL_MODEL_NAME}（首次可能会下载模型）")
    vectors, name = embed_with_local_model(texts)
    save_vector_cache(CACHE_PATH, vectors, name)
    return vectors, name

chunk_vectors, embedding_model_name = get_or_build_vectors()
print("当前 embedding:", embedding_model_name)


已加载向量缓存: chunk_vectors.npz  shape=(857, 1024)  model=api:qwen3-embedding-0.6b
当前 embedding: api:qwen3-embedding-0.6b


In [8]:
def embed_query(query, model_name):
    """问题必须和语料用同一个模型编码，否则余弦相似度没有意义。"""
    if model_name.startswith("api:"):
        vecs, _ = embed_with_api([query], model_name[len("api:"):])
        return vecs[0]
    if model_name == "unknown-cache":
        print(
            "警告：缓存里没有模型名。将按 BAAI/bge-small-zh-v1.5 编码问题。"
            "若第 1 题向量完全打不到第 18 页，说明缓存和当前模型不是同一套，"
            "请删掉 chunk_vectors.npz 后用同一模型重算。"
        )
    model = get_local_encoder()
    vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    return l2_normalize(vec)[0]


def search_vector_rows(query, top_k=3):
    q = embed_query(query, embedding_model_name)
    # 向量已 L2 归一化，点积就是余弦相似度
    scores = chunk_vectors @ q
    top_idx = np.argsort(scores)[::-1][:top_k]
    rows = []
    for i in top_idx:
        chunk = chunks[int(i)]
        rows.append({
            "idx": int(i),
            "score": float(scores[int(i)]),
            "chunk_id": chunk["chunk_id"],
            "pdf_page": chunk["pdf_page"],
            "text": chunk["text"],
        })
    return rows

print("向量检索函数已就绪。下一格先用第 1 题做一次环境核对。")
q1_vec = search_vector_rows(QUESTIONS[0], top_k=3)
print_rows("第 1 题 向量 top3（环境核对）", q1_vec)
if not any(row["chunk_id"] == "pumpkin-p018-c03" or row["pdf_page"] == 18 for row in q1_vec):
    print("注意：第 1 题的向量前 3 没有第 18 页。昨天原题是能打到的，请先检查是否用了同一份语料和同一个 embedding 模型。")
else:
    print("第 1 题向量仍能打到第 18 页，环境和昨天的对照基本一致。")


向量检索函数已就绪。下一格先用第 1 题做一次环境核对。
  API 编码进度 1/1
--- 第 1 题 向量 top3（环境核对） ---
  [1] score=0.633  pumpkin-p018-c03  pdf_page=18
  [2] score=0.620  pumpkin-p003-c02  pdf_page=3
  [3] score=0.603  pumpkin-p018-c01  pdf_page=18
第 1 题向量仍能打到第 18 页，环境和昨天的对照基本一致。


## 2. 本轮新概念：用排名融合，不要把分数加在一起

BM25 的分数可能是 19.3，向量相似度可能是 0.62。这两个数字不是同一把尺子，**直接相加没有意义**。那也不叫混合检索。

RRF（Reciprocal Rank Fusion，倒数排名融合）不管原始分数，只看名次：

```python
rrf_score += 1 / (60 + rank)  # rank 从 1 开始；某一路没出现就不加
```

可以把它想成：两个人分别给段落排序，再把名次换成可以相加的贡献值。

**演示数字**（为了讲清楚公式，不是昨天的完整 top10）：

- 片段甲（比如第 26 页，统计检验相关）：只在 BM25 排第 1 → `1/61 ≈ 0.0164`
- 片段乙（第 18 页，留出法 / 交叉验证 / 自助法）：只在向量排第 2 → `1/62 ≈ 0.0161`
- 片段丙：BM25 第 4、向量第 3 → `1/64 + 1/63 ≈ 0.0315`（融合后反而最高）

两路都找到的片段会往前排；某一路的第一名不一定稳赢。

做法：两路各取前 10，按 `chunk_id` 去重并累加贡献，再取融合后的前 3。融合没有提升也如实记录。


In [9]:
def rrf_fuse(list_of_rows, k=60, top_k=3):
    """用排名做融合。list_of_rows 里每一路都是已经按该路分数排好的列表。"""
    scores = {}
    meta = {}
    for rows in list_of_rows:
        for rank, row in enumerate(rows, start=1):
            cid = row["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
            # 保留第一次见到的原文和页码；融合分另外记录
            if cid not in meta:
                meta[cid] = dict(row)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    fused = []
    for cid, score in ranked:
        item = dict(meta[cid])
        item["score"] = score
        fused.append(item)
    return fused


def demo_rrf():
    bm25_demo = [
        {"chunk_id": "甲", "pdf_page": 26, "score": 0},
        {"chunk_id": "丁", "pdf_page": 28, "score": 0},
        {"chunk_id": "戊", "pdf_page": 27, "score": 0},
        {"chunk_id": "丙", "pdf_page": 19, "score": 0},
    ]
    vec_demo = [
        {"chunk_id": "己", "pdf_page": 3, "score": 0},
        {"chunk_id": "乙", "pdf_page": 18, "score": 0},
        {"chunk_id": "丙", "pdf_page": 19, "score": 0},
    ]
    fused = rrf_fuse([bm25_demo, vec_demo], k=60, top_k=3)
    print("演示：融合后的前 3（应看到丙的分数大约是 0.0315，并排在最前）")
    print_rows("RRF 演示", fused)

demo_rrf()


演示：融合后的前 3（应看到丙的分数大约是 0.0315，并排在最前）
--- RRF 演示 ---
  [1] score=0.031  丙  pdf_page=19
  [2] score=0.016  甲  pdf_page=26
  [3] score=0.016  己  pdf_page=3


上面这格跑完后，先看演示输出：丙同时出现在两路里，融合分大约是 `0.0315`，应排在第 1。若数字对得上，说明函数写对了，再跑真实四题。

## 3. 四题三路对照（本轮要跑完）

对每个问题：BM25 前 10、向量前 10，用 RRF（`k=60`）融合后取前 3，并和两路各自的前 3 并排打印。

重点看第 4 题：融合之后，第 18 页的 `pumpkin-p018-c03` 有没有进入前 3。


In [10]:
TARGET = "pumpkin-p018-c03"
RRF_K = 60
RETRIEVE_K = 10
FINAL_K = 3

all_results = []

for i, query in enumerate(QUESTIONS, start=1):
    print("=" * 70)
    print(f"问题 {i}: {query}")
    bm25_top10 = search_bm25_rows(query, top_k=RETRIEVE_K)
    vec_top10 = search_vector_rows(query, top_k=RETRIEVE_K)
    fused_top3 = rrf_fuse([bm25_top10, vec_top10], k=RRF_K, top_k=FINAL_K)
    print_rows(f"BM25 top{FINAL_K}", bm25_top10[:FINAL_K])
    print_rows(f"向量 top{FINAL_K}", vec_top10[:FINAL_K])
    print_rows(f"RRF 融合 top{FINAL_K} (k={RRF_K}, 各路先取 {RETRIEVE_K})", fused_top3)

    def rank_of(rows, chunk_id):
        for r, row in enumerate(rows, start=1):
            if row["chunk_id"] == chunk_id:
                return r
        return None

    print(
        f"  >> {TARGET} 的名次: "
        f"BM25={rank_of(bm25_top10, TARGET) or '未进前' + str(RETRIEVE_K)}  "
        f"向量={rank_of(vec_top10, TARGET) or '未进前' + str(RETRIEVE_K)}  "
        f"融合={rank_of(fused_top3, TARGET) or '未进前' + str(FINAL_K)}"
    )
    all_results.append(
        {
            "query": query,
            "bm25": bm25_top10,
            "vector": vec_top10,
            "rrf": fused_top3,
        }
    )

print("=" * 70)
print("四题跑完。下面一格会单独汇总第 4 题。")


问题 1: 书中介绍了哪三种模型评估方法？
  API 编码进度 1/1
--- BM25 top3 ---
  [1] score=19.305  pumpkin-p018-c01  pdf_page=18
  [2] score=18.562  pumpkin-p018-c03  pdf_page=18
  [3] score=17.303  pumpkin-p119-c03  pdf_page=119
--- 向量 top3 ---
  [1] score=0.633  pumpkin-p018-c03  pdf_page=18
  [2] score=0.620  pumpkin-p003-c02  pdf_page=3
  [3] score=0.603  pumpkin-p018-c01  pdf_page=18
--- RRF 融合 top3 (k=60, 各路先取 10) ---
  [1] score=0.033  pumpkin-p018-c03  pdf_page=18
  [2] score=0.032  pumpkin-p018-c01  pdf_page=18
  [3] score=0.016  pumpkin-p003-c02  pdf_page=3
  >> pumpkin-p018-c03 的名次: BM25=2  向量=1  融合=1
问题 2: 交叉验证法常用来比较哪些效果？
  API 编码进度 1/1
--- BM25 top3 ---
  [1] score=34.200  pumpkin-p018-c03  pdf_page=18
  [2] score=23.894  pumpkin-p019-c01  pdf_page=19
  [3] score=20.668  pumpkin-p019-c02  pdf_page=19
--- 向量 top3 ---
  [1] score=0.755  pumpkin-p019-c01  pdf_page=19
  [2] score=0.705  pumpkin-p018-c04  pdf_page=18
  [3] score=0.632  pumpkin-p019-c02  pdf_page=19
--- RRF 融合 top3 (k=60, 各路先取 10) ---


In [11]:
q4 = all_results[3]
print("第 4 题:", q4["query"])
print_rows("BM25 top3", q4["bm25"][:3])
print_rows("向量 top3", q4["vector"][:3])
print_rows("RRF 融合 top3", q4["rrf"])

ids_rrf = [row["chunk_id"] for row in q4["rrf"]]
print()
print(f"融合后，{TARGET}（第 18 页）是否进入前 3？", "是" if TARGET in ids_rrf else "否")
print("融合前 3 的 chunk_id:", ids_rrf)
print("请把这一格的输出贴回来，并自己写一句：融合有没有把第 18 页带进前 3。没有提升也如实说。")


第 4 题: 课本里用哪几种办法来检验模型表现如何？
--- BM25 top3 ---
  [1] score=16.884  pumpkin-p026-c02  pdf_page=26
  [2] score=14.787  pumpkin-p179-c02  pdf_page=179
  [3] score=12.916  pumpkin-p028-c01  pdf_page=28
--- 向量 top3 ---
  [1] score=0.650  pumpkin-p014-c05  pdf_page=14
  [2] score=0.634  pumpkin-p018-c03  pdf_page=18
  [3] score=0.620  pumpkin-p019-c03  pdf_page=19
--- RRF 融合 top3 ---
  [1] score=0.032  pumpkin-p014-c05  pdf_page=14
  [2] score=0.031  pumpkin-p026-c02  pdf_page=26
  [3] score=0.030  pumpkin-p016-c04  pdf_page=16

融合后，pumpkin-p018-c03（第 18 页）是否进入前 3？ 否
融合前 3 的 chunk_id: ['pumpkin-p014-c05', 'pumpkin-p026-c02', 'pumpkin-p016-c04']
请把这一格的输出贴回来，并自己写一句：融合有没有把第 18 页带进前 3。没有提升也如实说。


### 输出怎么读

- 先对一下第 1 题：两种单路检索都应该还能打到第 18 页，这是在确认环境和昨天一致。
- 再看第 4 题：重点不是「谁的分数更大」，而是 **目标证据在不在前 3**。
- 第 3 题即使三路都返回了片段，也不等于书里有「小明的哥哥」；有相似度只说明「比较像」，不是「有答案」。
- 融合分大约是 0.01～0.04 这种量级（因为是 `1/(60+名次)`），不要把它理解成正确概率。

**本轮结束时请交：** 第 4 题三路 top3 的输出，外加一句——融合后 `pumpkin-p018-c03` 有没有进入前 3。卡住了直接说卡在哪一格、报什么错。


## 4. 修改实验（下一轮再做，本轮先跳过）

下一轮只改一个地方：把 `RRF_K = 60` 改成 `RRF_K = 10`，只重新跑第 4 题，看 `pumpkin-p018-c03` 的融合名次变了没有。

`k` 更小，会更看重靠前的名次；`k` 更大，两路名次的差别会被抹得更平。先改再看结果，不必提前背结论。

更后面的轮次才会把融合后的前 3 条接回已有的 `--rag` 问答。本轮不要一次做完。


In [ ]:
# 下一轮再运行。本轮请先停在上面的第 4 题汇总。
# NEW_K = 10
# q = QUESTIONS[3]
# bm25_top10 = search_bm25_rows(q, top_k=10)
# vec_top10 = search_vector_rows(q, top_k=10)
# fused_k60 = rrf_fuse([bm25_top10, vec_top10], k=60, top_k=3)
# fused_k10 = rrf_fuse([bm25_top10, vec_top10], k=NEW_K, top_k=3)
# print_rows("RRF k=60", fused_k60)
# print_rows(f"RRF k={NEW_K}", fused_k10)
print("这是下一轮的修改实验，本轮不必运行。")
